In [33]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)   

#response = llm.invoke("Hola mundo")

#print(response)

In [34]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})


    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute(message)
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self, message):
        completion = llm.invoke(message)
        return completion
    

agente = Agent(system="Eres un asistente útil y objetivo.")
print(agente("Cuál es la capital de Argentina?"))

content=[{'type': 'text', 'text': 'La capital de Argentina es la **Ciudad Autónoma de Buenos Aires**.', 'extras': {'signature': 'EjQKMgERTTIPs2QMSKL214e8QAnDs1wxipSDdbK/yN9/uStsCUWB7FKlmynVL+bhXMEC2Qy3'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f5dc6-0554-7fd3-9750-2ecc9e7e25b0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 13, 'total_tokens': 21, 'input_token_details': {'cache_read': 0}}


In [ ]:
PROMPT_REACT = """
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.
Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas - y luego retorna "PAUSA". Usa el nombre exacto del elemento enviado por ejemplo: teclado de computadora
La "Observación" será el resultado de la acción ejecutada.
Acciones disponibles:

consultar_stock: devuelve la cantidad disponible de un articulo en el inventario (ej: "consultar_stock: teclado")
consultar_precio_producto: devuelve el precio unitario de un producto (ej: "consultar_precio_producto: mouse gamer")
encontrar_producto_mas_costoso: devuelve el nombre y el precio del producto más costoso del inventario (no requiere argumentos)
calcular_valor_total_lista: calcula el valor total de una lista de items de compra (ej: "calcular_valor_total_lista: teclado, mouse de gamer, monitor")

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en el inventario?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en el inventario.
Respuesta: Soy el agente: Hay 75 monitores en el inventario.
""".strip()

In [36]:
from langchain_protocol import TypedDict


class EstadoAgente(TypedDict):
    pregunta: str
    historial: list[str]
    accion_pendiente: str
    respuesta_final: str


def consultar_stock(item: str) -> str:
    """Simula la consulta de stock de item en el inventario."""

    item = item.lower()
    stock = {
        "monitor": 75,
        "teclado": 120,
        "mouse de gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impresora": 15,
    }

    if item in stock:
        return f"Tenemos {stock[item]} {item}s en stock."
    else:
        return f"Item '{item}' no encontrado en el inventario."


def consultar_precio_producto(producto: str) -> str:
    """Simula la consulta del precio unitario de un producto."""
    producto = producto.lower()
    precios = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00,
    }

    if producto in precios:
        return f"Soy fx: El precio de un(a) {producto} es USD {precios[producto]:.2f}."
    else:
        return f"producto '{producto}' no hallado en la lista de precios."


print(consultar_stock("teclado"))
print(consultar_precio_producto("impresora"))
print(consultar_stock("monitor"))
print(consultar_stock("sillas"))

Tenemos 120 teclados en stock.
Soy fx: El precio de un(a) impresora es USD 750.00.
Tenemos 75 monitors en stock.
Item 'sillas' no encontrado en el inventario.


In [37]:
def encontrar_producto_mas_costoso() -> str:
    """
    Retorna el nombre y el precio del producto más costoso en el inventario.
    Esta función no requiere argumentos adicionales.
    """
    precios_del_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00
    }
    
    if not precios_del_inventario:
        return "Lo sentimos, no hallamos ningún producto en la lista de precios para su comparación."

    nombre_producto_mas_costoso = max(precios_del_inventario, key=precios_del_inventario.get)
    valor_producto_mas_costoso = precios_del_inventario[nombre_producto_mas_costoso]
    
    return f"El producto más costoso es el(la) {nombre_producto_mas_costoso} con precio de USD {valor_producto_mas_costoso:.2f}."

In [46]:
def herramienta_calcular_valor_total_lista(lista_items: str) -> str:
    """
    Calcula el valor total de una lista de items de compra.
    Recibe una string con items separados por coma (ex: "teclado, mouse de gamer, monitor").
    """

    precios_del_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00,
    }

    items_procesados = [item.strip().lower() for item in lista_items.split(',')]

    valor_total = 0.0
    items_no_encontrados = []

    for item in items_procesados:
        if item in precios_del_inventario:
            valor_total += precios_del_inventario[item]
        else:
            items_no_encontrados.append(item)

    respuesta = f"El valor total de los items encontrados es USD {valor_total:.2f}."
    if items_no_encontrados:
        respuesta += f" Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: {', '.join(items_no_encontrados)}."

    return respuesta

In [47]:
import re


def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
    current_prompt = f"{PROMPT_REACT}\n\nPregunta: {pregunta}"

    for i in range(max_iterations):
        response = llm.invoke(current_prompt)
        response_text = getattr(response, "text", getattr(response, "content", str(response))).strip()

        print(f"--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")

        if "Respuesta:" in response_text:
            return response_text.split("Respuesta:", 1)[1].strip()

        match = re.search(r"Acción:\s*([A-Za-z_][A-Za-z0-9_]*)\s*(?:\((.*)\)|:(.*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = (match.group(2) or match.group(3) or "").strip()

            observacion = ""
            if action_name == "consultar_stock":
                observacion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion = consultar_precio_producto(action_arg)
            elif action_name == "encontrar_producto_mas_costoso":
                observacion = encontrar_producto_mas_costoso()
            elif action_name == "calculador_valor_total_lista":
                observacion = herramienta_calcular_valor_total_lista(action_arg)
            else:
                observacion = f"Error: Acción '{action_name}' desconocida."

            current_prompt = f"{current_prompt}\n\nObservación: {observacion}\nRespuesta:"

            print(f"Ejecutó acción: {action_name}({action_arg})")
            print(f"Observación: {observacion}\n")
        else:
            return f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La última respuesta fue: {response_text}"

    return "Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

In [50]:
print("Testando herramienta_calcular_valor_total_lista:")

lista_1 = "teclado, mouse de gamer, monitor"
resultado_1 = herramienta_calcular_valor_total_lista(lista_1)
print(f"Lista: '{lista_1}'\nResultado: {resultado_1}\n")

lista_2 = "headset, silla"
resultado_2 = herramienta_calcular_valor_total_lista(lista_2)
print(f"Lista: '{lista_2}'\nResultado: {resultado_2}\n")

lista_3 = "mesa, vaso"
resultado_3 = herramienta_calcular_valor_total_lista(lista_3)
print(f"Lista: '{lista_3}'\nResultado: {resultado_3}\n")

Testando herramienta_calcular_valor_total_lista:
Lista: 'teclado, mouse de gamer, monitor'
Resultado: El valor total de los items encontrados es USD 1249.40.

Lista: 'headset, silla'
Resultado: El valor total de los items encontrados es USD 180.00. Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: silla.

Lista: 'mesa, vaso'
Resultado: El valor total de los items encontrados es USD 0.00. Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: mesa, vaso.



In [ ]:
pregunta_4 = "Cuál es el producto más costoso?"
print(f"**Interacción 4: {pregunta_4}**")
respuesta_4 = run_react_agent(pregunta_4)
print(f"\n**RESPUESTA FINAL DEL AGENTE 4:** {respuesta_4}\n")

**Interacción 4: Cuál es el producto más costoso?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo utilizar la herramienta `encontrar_producto_mas_costoso` para determinar cuál es el artículo con el precio más alto en el inventario.

Acción: encontrar_producto_mas_costoso
PAUSA

Ejecutó acción: encontrar_producto_mas_costoso()
Observación: El producto más costoso es el(la) monitor con precio de USD 999.90.

--- Iteración 2 ---
Modelo pensó/respondió:
Pensamiento: Debo utilizar la herramienta encontrar_producto_mas_costoso para obtener la información solicitada.

Acción: encontrar_producto_mas_costoso
PAUSA

Observación: El producto más costoso es el monitor con un precio de USD 999.90.

Respuesta: Soy el agente: El producto más costoso del inventario es el monitor, con un precio de USD 999.90.


**RESPUESTA FINAL DEL AGENTE 4:** Soy el agente: El producto más costoso del inventario es el monitor, con un precio de USD 999.90.



In [ ]:
pregunta_1 = "Cuántos \"mouse de gamer\" están disponibles en el inventario?"
print(f"**Interacción 1: {pregunta_1}**")
respuesta_1 = run_react_agent(pregunta_1)
print(f"\n**RESPUESTA FINAL DEL AGENTE 1:** {respuesta_1}\n")

print("\n" + "="*50 + "\n")

**Interacción 1: Cuántos "mouse de gamer" están disponibles en el inventario?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la cantidad de "mouse de gamer" en el inventario utilizando la herramienta correspondiente.

Acción: consultar_stock: mouse de gamer
PAUSA

Ejecutó acción: consultar_stock(mouse de gamer)
Observación: Tenemos 80 mouse de gamers en stock.

--- Iteración 2 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de "mouse de gamer" en el inventario.
Acción: consultar_stock: mouse de gamer
PAUSA

Observación: Tenemos 80 mouse de gamers en stock.
Respuesta: Soy el agente: Hay 80 mouse de gamer disponibles en el inventario.


**RESPUESTA FINAL DEL AGENTE 1:** Soy el agente: Hay 80 mouse de gamer disponibles en el inventario.





In [ ]:
pregunta_2 = "Cuál es el precio de una impresora?"
print(f"**Interacción 2: {pregunta_2}**")
respuesta_2 = run_react_agent(pregunta_2)
print(f"\n**RESPUESTA FINAL DEL AGENTE 2:** {respuesta_2}\n")

print("\n" + "="*50 + "\n")

**Interacción 2: Cuál es el precio de una impresora?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_precio_producto para conocer el valor de una impresora.
Acción: consultar_precio_producto: impresora
PAUSA

Ejecutó acción: consultar_precio_producto(impresora)
Observación: Soy fx: El precio de un(a) impresora es USD 750.00.

--- Iteración 2 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar el precio de la impresora utilizando la herramienta consultar_precio_producto.

Acción: consultar_precio_producto: impresora
PAUSA

Observación: El precio de una impresora es USD 750.00.

Respuesta: Soy fx: El precio de una impresora es USD 750.00.


**RESPUESTA FINAL DEL AGENTE 2:** Soy fx: El precio de una impresora es USD 750.00.





In [ ]:
pregunta_3 = "Tenemos sillas en inventario?"
print(f"**Interacción 3: {pregunta_3}**")
respuesta_3 = run_react_agent(pregunta_3)
print(f"\n**RESPUESTA FINAL DEL AGENTE 3:** {respuesta_3}\n")

**Interacción 3: Tenemos sillas en inventario?**
--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_stock para verificar la disponibilidad de sillas en el inventario.

Acción: consultar_stock: sillas
PAUSA

Ejecutó acción: consultar_stock(sillas)
Observación: Item 'sillas' no encontrado en el inventario.

--- Iteración 2 ---
Modelo pensó/respondió:
Pensamiento: Debo verificar si el producto "sillas" existe en el inventario utilizando la herramienta consultar_stock.

Acción: consultar_stock: sillas
PAUSA

Observación: Item 'sillas' no encontrado en el inventario.

Respuesta: Soy el agente: Lo siento, no tenemos sillas en el inventario.


**RESPUESTA FINAL DEL AGENTE 3:** Soy el agente: Lo siento, no tenemos sillas en el inventario.

